## ARMA


parquet data, 4 channels, all subjects, 5 separate windows(ctx=512, h=64)

In [6]:
!pip install pyarrow pandas scikit-learn scipy statsmodels -q

## Config

In [7]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from scipy.stats import zscore
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', UserWarning)

KAGGLE_DATASET_DIR = "/kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw"
PARQUET_FILENAME = "eeg_preprocessed_4ch_raw.parquet"
DATA_PATH = os.path.join(KAGGLE_DATASET_DIR, PARQUET_FILENAME)
PARQUET_PATH = os.environ.get("EEG_PARQUET", DATA_PATH)

LIMIT_PATIENTS = None
CHANNELS = ["Fp1", "Fp2", "P3", "P4"]
CONTEXT_LENGTH = 512
HORIZON_LENGTH = 64
N_WINDOWS = 5
FS = 500 # Native sampling rate
OFFSET_SAMPLES = 3 * FS

ARMA_ORDER = (2, 0, 2) 

df = pd.read_parquet(PARQUET_PATH)
df_eval = df.head(LIMIT_PATIENTS).copy() if LIMIT_PATIENTS else df.copy()
n_total = len(df)
print(
    f"Loaded parquet: {PARQUET_PATH} | rows in file: {n_total}, "
    f"evaluation on (the first) {len(df_eval)} subjects."
)

Loaded parquet: /kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw/eeg_preprocessed_4ch_raw.parquet | rows in file: 88, evaluation on (the first) 88 subjects.


## Model

In [8]:
CSV_OUT = "arma_eeg_micro_eval_results.csv"

def forecast_windows(normed_signal, starts, ctx_len, hor_len, order):
    preds_all = []
    
    for s in starts:
        # Wycinamy znormalizowany kontekst (dane wejściowe do nauki)
        ctx_numpy = normed_signal[s : s + ctx_len]
        
        # Inicjalizacja i dopasowanie (trening) modelu ARMA dla pojedynczego okna
        model = ARIMA(ctx_numpy, order=order)
        fitted_model = model.fit()
        
        # Predykcja na żądany horyzont
        pred = fitted_model.forecast(steps=hor_len)
        preds_all.append(pred)
        
    return np.array(preds_all)

## Processing functions

In [9]:
def window_starts(total_len: int, ctx: int, hor: int, n_win: int, offset: int = OFFSET_SAMPLES) -> np.ndarray:
    need = ctx + hor
    available_len = total_len - offset
    if available_len < need:
        raise ValueError(f"Signal too short: available={available_len}, required >= {need}")
    hi = total_len - need
    
    return np.linspace(offset, hi, n_win, dtype=int)

def series_to_numpy(cell) -> np.ndarray:
    if isinstance(cell, (list, np.ndarray)):
        return np.asarray(cell, dtype=np.float64)
    return np.asarray(cell.tolist() if hasattr(cell, "tolist") else cell, dtype=np.float64)

## Evaluation loop

In [10]:
detail_rows = []
done = 0

for _, row in df_eval.iterrows():
    sid = row["subject_id"]
    grp = row["group"] if "group" in row.index and pd.notna(row["group"]) else "Unknown"
    
    for ch in CHANNELS:
        raw_signal = series_to_numpy(row[ch])
        
        raw_std = float(np.std(raw_signal))
        normed_signal = zscore(raw_signal).astype(np.float64) # statsmodels preferuje float64
        
        starts = window_starts(len(normed_signal), CONTEXT_LENGTH, HORIZON_LENGTH, N_WINDOWS)
            
        preds = forecast_windows(
            normed_signal=normed_signal, 
            starts=starts, 
            ctx_len=CONTEXT_LENGTH, 
            hor_len=HORIZON_LENGTH,
            order=ARMA_ORDER
        )
        
        # Obliczanie błędów (MSE normowane i fizyczne)
        mses_win = []
        for i, s in enumerate(starts):
            tgt_norm = normed_signal[s + CONTEXT_LENGTH : s + CONTEXT_LENGTH + HORIZON_LENGTH]
            mses_win.append(mean_squared_error(tgt_norm, preds[i]))
            
        mse_norm = float(np.mean(mses_win))
        mse_phys = mse_norm * (raw_std ** 2) 
        
        detail_rows.append(
            {
                "record_type": "per_patient_electrode",
                "subject_id": sid,
                "group": grp,
                "electrode": ch,
                "mse_norm": mse_norm, 
                "mse_phys": mse_phys, 
                "n_windows": N_WINDOWS,
            }
        )
    done += 1
    if done % 10 == 0 or done == len(df_eval):
        print(f"Subjects analysed: {done} / {len(df_eval)} (last: {sid}).")

# --- Zbieranie statystyk i zapis (identyczne jak w poprzednim notatniku) ---
df_detail = pd.DataFrame(detail_rows)

summary_rows = []
for g in sorted(df_detail["group"].dropna().unique()):
    sub = df_detail[(df_detail["record_type"] == "per_patient_electrode") & (df_detail["group"] == g)]
    if sub.empty:
        continue
    
    summary_rows.append(
        {
            "record_type": "group_all_electrodes",
            "subject_id": "",
            "group": g,
            "electrode": "ALL",
            "mse_norm": float(sub["mse_norm"].mean()),
            "mse_phys": float(sub["mse_phys"].mean()),
            "n_windows": N_WINDOWS,
        }
    )
    for ch in CHANNELS:
        sub_ch = sub[sub["electrode"] == ch]
        if sub_ch.empty:
            continue
        summary_rows.append(
            {
                "record_type": "group_per_electrode",
                "subject_id": "",
                "group": g,
                "electrode": ch,
                "mse_norm": float(sub_ch["mse_norm"].mean()),
                "mse_phys": float(sub_ch["mse_phys"].mean()),
                "n_windows": N_WINDOWS,
            }
        )

df_out = pd.concat([df_detail, pd.DataFrame(summary_rows)], ignore_index=True)
df_out.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT}")

try:
    from IPython.display import display
    display(df_out[['record_type', 'subject_id', 'group', 'electrode', 'mse_norm', 'mse_phys']].tail(20))
except ImportError:
    print(df_out.tail(20).to_string())

Subjects analysed: 10 / 88 (last: sub-010).
Subjects analysed: 20 / 88 (last: sub-020).
Subjects analysed: 30 / 88 (last: sub-030).
Subjects analysed: 40 / 88 (last: sub-040).
Subjects analysed: 50 / 88 (last: sub-050).
Subjects analysed: 60 / 88 (last: sub-060).
Subjects analysed: 70 / 88 (last: sub-070).
Subjects analysed: 80 / 88 (last: sub-080).
Subjects analysed: 88 / 88 (last: sub-088).
Saved: arma_eeg_micro_eval_results.csv


,record_type,subject_id,group,electrode,mse_norm,mse_phys
347,per_patient_electrode,sub-087,F,P4,0.408102,4.380355e-10
348,per_patient_electrode,sub-088,F,Fp1,0.750889,7.650047e-10
349,per_patient_electrode,sub-088,F,Fp2,0.825314,8.530080e-10
350,per_patient_electrode,sub-088,F,P3,0.511518,5.097505e-10
351,per_patient_electrode,sub-088,F,P4,0.596586,6.050334e-10
352,group_all_electrodes,,A,ALL,0.664773,7.718921e-10
353,group_per_electrode,,A,Fp1,0.686039,8.209409e-10
354,group_per_electrode,,A,Fp2,0.681793,8.525828e-10
355,group_per_electrode,,A,P3,0.640762,7.010520e-10
356,group_per_electrode,,A,P4,0.650498,7.129929e-10
